![](../img/logo_ucm.jpg)

# Ejercicio de cierre tema 2: microservicio de inferencia con serialización

En este ejercicio vas a combinar los conceptos trabajados en los notebooks de **FastAPI** y **serialización de modelos**. El objetivo es entrenar un modelo sencillo con `iris`, guardarlo a disco con `pickle`, cargarlo desde una API y exponer un endpoint REST para hacer predicciones.


## Objetivos

1. Entrenar un clasificador sobre el dataset `iris`.
2. Serializar el modelo entrenado en un fichero `.pkl`.
3. Deserializar el modelo para servirlo desde FastAPI.
4. Definir un esquema de entrada con Pydantic.
5. Exponer dos endpoints:
   - `GET /health`
   - `POST /predict`
6. Consumir la API desde un cliente Python con `requests`.

## Requisitos funcionales

- El endpoint `/predict` debe recibir un JSON con las cuatro variables del iris.
- La API debe devolver la especie predicha.
- Debe existir al menos una validación de datos de entrada.
- El modelo servido debe cargarse desde disco, no reutilizar directamente la variable entrenada en memoria.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle
import threading

import nest_asyncio
import pandas as pd
import requests
import seaborn as sns
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, field_validator
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

MODEL_DIR = Path("artifacts")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "iris_api_model_v01.pkl"
FEATURE_NAMES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
EXERCISE_PORT = 8001


## Apartado 1: entrenamiento del modelo

Carga el dataset `iris`, selecciona las cuatro columnas predictoras y entrena un `DecisionTreeClassifier` con `random_state=99`. Calcula la métrica `accuracy` sobre test.


In [ ]:
iris = sns.load_dataset("iris")
iris.sample(5, random_state=99)

# TODO (1): seleccionar features y target
# TODO (2): dividir en train y test con test_size=0.2 y random_state=99
# TODO (3): entrenar un DecisionTreeClassifier(random_state=99)
# TODO (4): calcular accuracy en test e imprimirlo

X = ...
y = ...

X_train, X_test, y_train, y_test = ...

model = ...
...

accuracy = ...
print(f"Accuracy en test: {accuracy:.4f}")


## Apartado 2: serialización y carga del modelo

Guarda el modelo con `pickle` en `artifacts/iris_api_model_v01.pkl`. Después vuelve a cargarlo en otra variable y verifica que predice correctamente para un ejemplo.


In [ ]:
# TODO (5): serializar el modelo entrenado usando pickle
with open(MODEL_PATH, "wb") as file:
    ...

# TODO (6): deserializar el modelo en una nueva variable
with open(MODEL_PATH, "rb") as file:
    loaded_model = ...

example_df = pd.DataFrame([[6.9, 3.1, 5.4, 2.1]], columns=FEATURE_NAMES)
print("Predicción de prueba:", loaded_model.predict(example_df)[0])


## Apartado 3: contrato de entrada con Pydantic

Define una clase `IrisFeatures` con las cuatro variables de entrada y añade una validación para garantizar que todos los valores sean positivos.


In [ ]:
class IrisFeatures(BaseModel):
    # TODO (7): definir los cuatro atributos float
    ...

    # TODO (8): añadir una validación para comprobar que todos los campos sean positivos
    ...


## Apartado 4: microservicio con FastAPI

Construye una API que cargue el modelo desde disco y exponga:

- `GET /health`: devuelve un diccionario con el estado del servicio y si el modelo está cargado.
- `POST /predict`: recibe un objeto `IrisFeatures`, transforma los datos a `DataFrame` y devuelve la especie predicha.


In [ ]:
app = FastAPI(title="Iris Microservice Exercise")

# TODO (9): cargar el modelo serializado desde disco
with open(MODEL_PATH, "rb") as file:
    deployed_model = ...

@app.get("/health")
def healthcheck():
    # TODO (10): devolver el estado del servicio
    ...

@app.post("/predict")
def predict(features: IrisFeatures):
    # TODO (11): convertir el objeto Pydantic a diccionario
    # TODO (12): construir el DataFrame de entrada usando FEATURE_NAMES
    # TODO (13): llamar a predict sobre deployed_model
    # TODO (14): devolver un diccionario con la predicción
    ...


## Apartado 5: levantar el servidor

Ejecuta la siguiente celda para lanzar el servicio en segundo plano dentro del notebook. Si el puerto ya estuviera ocupado, cambia `EXERCISE_PORT` en la celda de imports.


In [ ]:
nest_asyncio.apply()

def run():
    uvicorn.run(app, host="127.0.0.1", port=EXERCISE_PORT)

thread = threading.Thread(target=run)
thread.start()

print(f"Servidor levantado en http://127.0.0.1:{EXERCISE_PORT}")


## Apartado 6: consumo de la API

Haz una petición a `/health` y otra a `/predict` usando `requests`. Para la predicción, utiliza como ejemplo los valores `6.9, 3.1, 5.4, 2.1`.


In [ ]:
base_url = f"http://127.0.0.1:{EXERCISE_PORT}"

# TODO (15): llamar al endpoint /health y mostrar la respuesta JSON
health_response = ...
print("Health:", health_response.json())

payload = {
    "sepal_length": 6.9,
    "sepal_width": 3.1,
    "petal_length": 5.4,
    "petal_width": 2.1,
}

# TODO (16): llamar al endpoint /predict con un POST enviando el payload en formato JSON
prediction_response = ...
print("Predict:", prediction_response.json())


## Preguntas

1. ¿Qué ventaja aporta cargar el modelo desde disco frente a reutilizar directamente la variable `model` entrenada en memoria?
2. ¿Qué error devuelve FastAPI si falta un campo del JSON o si el tipo no coincide?
3. ¿Qué limitaciones tiene `pickle` para un despliegue serio en producción?
4. ¿Qué cambiarías si quisieras servir varias versiones del modelo a la vez?
